# Panel de información adicional / estructura

Apila los archivos `inf_adi/{codigo}.txt` de los 133 dumps IEF. Trae cantidad de cuentas (corrientes, cajas de ahorro, plazos fijos), préstamos por tipo, tarjetas, dotación de personal, cantidad de clientes, etc. Cinco fechas por dump.

Las 5 fechas son las mismas que las del balance (`entidad/{cod}.txt` cols 13-17).

Output: `data/interim/paneles_largos/panel_estructura.parquet`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_estructura.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

INF_FECHA_COLS = ["v1", "v2", "v3", "v4", "v5"]
ENTIDAD_FECHA_IDX = list(range(12, 17))  # mismas fechas que balance

## Helpers

In [ ]:
def leer_fechas_balance(entidad_file):
    texto = entidad_file.read_text(encoding="ISO-8859-1")
    campos = [c.strip().strip('"') for c in texto.split("\t")]
    if len(campos) <= ENTIDAD_FECHA_IDX[-1]:
        return []
    return [campos[i] for i in ENTIDAD_FECHA_IDX]

def leer_inf_adi(inf_file, fechas_5):
    cols = ["codigo_entidad", "nombre_entidad", "fecha_corte", "codigo_linea",
            "descripcion_informacion", "v1", "v2", "v3", "v4", "v5"]
    df = pd.read_csv(inf_file, sep="\t", header=None, names=cols,
                     encoding="ISO-8859-1", dtype=str)
    for c in INF_FECHA_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    mapeo = dict(zip(INF_FECHA_COLS, fechas_5))
    long = df.melt(
        id_vars=["codigo_entidad", "nombre_entidad", "codigo_linea", "descripcion_informacion"],
        value_vars=INF_FECHA_COLS, var_name="slot_fecha", value_name="valor"
    )
    long["yyyymm"] = long["slot_fecha"].map(mapeo)
    long = long.drop(columns=["slot_fecha"]).dropna(subset=["yyyymm"])
    return long

## Apilado

In [ ]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
bloques = []
for d in dumps:
    yyyymm_dump = d.name
    inf_dir = d / "Entfin/Tec_Cont/inf_adi"
    entidad_dir = d / "Entfin/Tec_Cont/entidad"
    if not inf_dir.exists() or not entidad_dir.exists():
        continue
    for inf_file in inf_dir.glob("*.txt"):
        if inf_file.name == "formato.txt":
            continue
        cod = inf_file.stem
        entidad_file = entidad_dir / f"{cod}.txt"
        if not entidad_file.exists():
            continue
        try:
            fechas = leer_fechas_balance(entidad_file)
            if len(fechas) != 5:
                continue
            long = leer_inf_adi(inf_file, fechas)
            long["dump_yyyymm"] = int(yyyymm_dump)
            bloques.append(long)
        except Exception:
            pass

raw = pd.concat(bloques, ignore_index=True)
print(f"Filas apiladas: {len(raw):,}")

## Deduplicación y persistencia

In [ ]:
raw = raw[raw["yyyymm"].astype(str).str.match(r"^\d{6}$")].copy()
raw["yyyymm"] = raw["yyyymm"].astype(int)
raw = raw.sort_values(["codigo_entidad", "yyyymm", "codigo_linea", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "yyyymm", "codigo_linea"], keep="last")
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel[["codigo_entidad", "yyyymm", "codigo_linea"]].duplicated().sum() == 0
print("Validaciones OK")
print(f"  entidades: {panel['codigo_entidad'].nunique()}")
print(f"  variables distintas: {panel['codigo_linea'].nunique()}")
print(f"  rango fechas: {panel['yyyymm'].min()} - {panel['yyyymm'].max()}")

## Muestra: variables disponibles

In [ ]:
duckdb.sql(f"""
    select distinct codigo_linea, descripcion_informacion
    from '{OUT}'
    order by codigo_linea
    limit 20
""").df()